In [4]:
import random
import pandas as pd
import numpy as np
from datetime import datetime
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
# 假設 persona.py 已定義，包含 initial_stance (float: -1.0 ~ 1.0)
from persona import generate_persona 

# === 1. 研究方向 4：量化評估指標計算 ===
def calculate_polarization(stances):
    """計算群體極化程度：標準差越大，代表意見越分裂"""
    return np.std(stances)

def calculate_consensus(stances):
    """計算共識度：平均值偏向某一極端且標準差小，代表形成規範"""
    avg = np.mean(stances)
    std = np.std(stances)
    return avg, 1.0 / (1.0 + std)

# === 2. 研究方向 2：MCP 工具實作 (模擬干預策略) ===
class MCPToolbox:
    @staticmethod
    def fact_check(query, intervention_type="None"):
        """
        模擬不同強度的干預策略：
        - None: 一般事實
        - High_Visibility: 強制性真相推送 (降低謠言再生率)
        """
        base_info = "【MCP 權威庫】: eID 晶片符合 ISO 14443 標準，採去中心化驗證。"
        if intervention_type == "High_Visibility":
            return f"{base_info} (警告：此訊息已由第三方核實，請勿傳播未經證實之謠言)"
        return base_info

# === 3. 核心模擬引擎 ===
class ReprodSocialSim:
    def __init__(self, intervention_strategy="None"):
        self.logs = []
        self.conversation_memory = []
        self.llm = ChatOllama(model="llama3:8B", temperature=0.7)
        self.intervention = intervention_strategy
        self.agent_stances = {} # 追蹤每個 Agent 的數值化立場

    def analyze_stance_change(self, text):
        """
        模擬研究方向 3：量化立場變化
        實務上可再串接一個小模型進行 NLI 或 Sentiment Analysis
        這裡簡化模擬立場值回傳 (-1.0 反對, 0.0 中立, 1.0 支持)
        """
        if "support" in text.lower() or "agree" in text.lower(): return 0.2
        if "skeptical" in text.lower() or "reservation" in text.lower(): return -0.2
        return 0.0

    def run_round(self, agents, round_num):
        round_data = []
        names = list(agents.keys())
        random.shuffle(names)

        for name in names:
            persona = agents[name]
            # 動機驅動的工具使用：當 Agent 感到疑惑或遇到衝突時使用 MCP
            use_tool = random.random() > 0.4 
            mcp_info = MCPToolbox.fact_check(TOPIC, self.intervention) if use_tool else "無"

            prompt = ChatPromptTemplate.from_messages([
                ("system", f"{persona.to_prompt()}\n當前社會干預狀態：{self.intervention}"),
                ("human", f"第 {round_num} 輪討論。\n話題：{TOPIC}\nMCP 參考：{mcp_info}\n記憶：{self.conversation_memory[-3:]}\n請回應：")
            ])

            chain = prompt | self.llm
            response = chain.invoke({}).content.strip()
            
            # 更新立場 (模擬規範形成速度)
            shift = self.analyze_stance_change(response)
            self.agent_stances[name] = np.clip(self.agent_stances[name] + shift, -1.0, 1.0)

            self.conversation_memory.append(f"{name}: {response}")
            round_data.append({
                "round": round_num,
                "agent": name,
                "stance_score": self.agent_stances[name],
                "mcp_used": use_tool,
                "text": response
            })
        return round_data

    def start_experiment(self, num_agents=5, rounds=3):
        # 初始化
        agents = {f"User_{i}": generate_persona() for i in range(num_agents)}
        self.agent_stances = {name: random.uniform(-0.5, 0.5) for name in agents}

        for r in range(1, rounds + 1):
            print(f"執行第 {r} 輪...")
            results = self.run_round(agents, r)
            
            # 計算該輪量化指標 (研究方向 4)
            current_stances = list(self.agent_stances.values())
            pol = calculate_polarization(current_stances)
            avg_stance, cons = calculate_consensus(current_stances)
            
            for entry in results:
                entry["group_polarization"] = pol
                entry["group_consensus"] = cons
                entry["intervention"] = self.intervention
                self.logs.append(entry)

        # 匯出 CSV
        df = pd.DataFrame(self.logs)
        filename = f"exp_{self.intervention}_{datetime.now().strftime('%m%d%H%M')}.csv"
        df.to_csv(filename, index=False)
        return df

# === 4. 目的 3：系統性評估干預效果 ===
if __name__ == "__main__":
    TOPIC = "數位身分證隱私爭議"
    
    # 實驗組 1: 無干預 (對照組)
    sim_control = ReprodSocialSim(intervention_strategy="None")
    df_control = sim_control.start_experiment()

    # 實驗組 2: 強力 MCP 事實干預 (實驗組)
    sim_test = ReprodSocialSim(intervention_strategy="High_Visibility")
    df_test = sim_test.start_experiment()
    
    print("實驗完成。您可以比較兩組 CSV 中的 'group_polarization' 變化速度。")

執行第 1 輪...
執行第 2 輪...
執行第 3 輪...
執行第 1 輪...
執行第 2 輪...
執行第 3 輪...
實驗完成。您可以比較兩組 CSV 中的 'group_polarization' 變化速度。


MCP 工具深度集成 (研究方向 2)：

原本代碼中 MCP 只是隨機字串。新架構中，MCPToolbox 可以根據「干預策略」提供不同品質的資訊。

這符合您「評估干預策略效果」的目的 。

量化評估機制 (研究方向 3 & 4)：

極化 (Polarization)：透過計算 Agent 立場分數的「標準差」來衡量。

規範形成速度：觀察 stance_score 隨輪次收斂至特定區間的速度。

謠言再生率：可透過統計 Agent 在接收 MCP 正確資訊後，下一輪是否仍產出「錯誤資訊（反對立場）」來估算。

可重現性設計 (目的 1 & 2)：

透過將 intervention_strategy 參數化，您可以針對同一組 Persona 重複執行實驗，驗證在不同干預下，平台是否能穩定重建（Reconstruct）典型的社會現象（如：加入強干預後極化程度下降） 。

立場追蹤 (Stance Tracking)：

新增了 agent_stances 字典與 analyze_stance_change 函數，這讓 CSV 數據不再只是純文字，而是包含可以繪製趨勢圖的數值 。